# 11 — CatBoost

**Algorithm:** CatBoost (Categorical Boosting)  
**Features:** TF-IDF (5K) + TruncatedSVD-200 (dense features required)  
CatBoost uses symmetric trees and ordered boosting to reduce prediction shift bias. Dense features are required as CatBoost does not natively handle scipy sparse matrices.

In [ ]:
import os, sys, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))
from utils import (setup_paths, evaluate_and_print, plot_confusion_matrix,
                   plot_roc_curve, plot_pr_curve, plot_learning_curve, get_dense_features)
BASE_DIR, FIGURES_DIR = setup_paths()

try:
    from catboost import CatBoostClassifier, Pool
    import catboost
    print(f'CatBoost version: {catboost.__version__}')
except ImportError:
    raise ImportError('CatBoost is not installed. Run: pip install catboost')

In [ ]:
from preprocess import load_all_datasets, get_train_test_split
df = load_all_datasets()
X_train, X_test, y_train, y_test = get_train_test_split(df)
print(f'Train: {len(X_train):,}  |  Test: {len(X_test):,}')

In [ ]:
X_tr_d, X_te_d, dense_idx = get_dense_features(X_train, X_test, max_features=5000, n_components=200, sample_size=20000)
y_tr_d = y_train[dense_idx] if dense_idx is not None else y_train
print(f'Dense features shape: Train={X_tr_d.shape}  Test={X_te_d.shape}')

## Model Training

In [ ]:
MODEL_NAME = 'CatBoost'
train_pool = Pool(X_tr_d, y_tr_d)
test_pool  = Pool(X_te_d, y_test)

model = CatBoostClassifier(
    iterations=100, random_seed=42, verbose=0,
    eval_metric='Logloss', use_best_model=True
)
t0 = time.time()
model.fit(train_pool, eval_set=test_pool, verbose=False)
train_time = time.time() - t0
print(f'Training completed in {train_time:.2f}s')
print(f'Best iteration: {model.get_best_iteration()}')

In [ ]:
metrics, y_pred, y_prob = evaluate_and_print(model, X_te_d, y_test, MODEL_NAME, train_time)

In [ ]:
plot_confusion_matrix(y_test, y_pred, MODEL_NAME, FIGURES_DIR)

In [ ]:
plot_roc_curve(y_test, y_prob, MODEL_NAME, FIGURES_DIR)

In [ ]:
plot_pr_curve(y_test, y_prob, MODEL_NAME, FIGURES_DIR)

## Training Convergence

In [ ]:
evals = model.get_evals_result()
tr_loss = evals.get('learn', {}).get('Logloss', [])
te_loss = evals.get('validation', {}).get('Logloss', [])

if tr_loss and te_loss:
    iters = np.arange(1, len(tr_loss) + 1)
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(iters, tr_loss, color='#3498db', lw=2, label='Train Log-Loss')
    ax.plot(iters, te_loss, color='#e74c3c', lw=2, label='Test Log-Loss')
    best_it = model.get_best_iteration()
    if best_it:
        ax.axvline(best_it, color='green', linestyle='--', lw=1.5,
                   label=f'Best iter = {best_it}')
    ax.set_xlabel('Iteration')
    ax.set_ylabel('Log-Loss')
    ax.set_title('CatBoost — Training Convergence')
    ax.legend(fontsize=10); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURES_DIR, 'catboost_convergence.png'), dpi=150, bbox_inches='tight')
    plt.show()

## Feature Importances

In [ ]:
fi = model.get_feature_importance()
top_n = min(20, len(fi))
top_idx = np.argsort(fi)[-top_n:][::-1]
fig, ax = plt.subplots(figsize=(9, 5))
ax.barh([f'SVD-{i}' for i in top_idx[::-1]], fi[top_idx[::-1]], color='#c0392b', alpha=0.85)
ax.set_xlabel('Feature Importance')
ax.set_title('CatBoost — Top Feature Importances')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'catboost_feature_importance.png'), dpi=150, bbox_inches='tight')
plt.show()

## Learning Curve

In [ ]:
plot_learning_curve(model, X_tr_d, y_tr_d, X_te_d, y_test, MODEL_NAME, FIGURES_DIR)